# CRISP-DM Phase 3 — Data Preparation

**Project:** ML4B SoSe 2026 — Gym Exercise Recognition from Apple Watch Sensor Data  
**Dataset:** RecoFit (Microsoft Research, Morris et al., CHI 2014)  
**Phase goal:** Turn the raw 2.5 GB `.mat` file into a clean, feature-engineered, subject-disjoint train/val/test CSV trio that downstream modelling notebooks can consume directly.

## Pipeline overview

1. **Load** raw recordings → long-format DataFrame (`src/ml4b/data/loader.py`)  
2. **Window** continuous signals → fixed 2-second windows with 50% overlap (`src/ml4b/data/windowing.py`)  
3. **Extract features** → per-window statistical + FFT feature vector (`src/ml4b/data/features.py`)  
4. **Split** by subject → train / val / test, no subject overlap (`src/ml4b/data/splitting.py`)  
5. **Save** to `data/processed/` as CSVs ready for Phase 4 modelling.

All pipeline logic lives in the reusable `src/ml4b/` package — this notebook is just the orchestration and sanity-check layer.

## 1. Imports and Project Root

`find_project_root()` walks up from the notebook's working directory until it finds `pyproject.toml`. This works regardless of where Jupyter is launched (WSL, macOS, Windows, VS Code). All file paths are resolved relative to that root via constants from `ml4b.utils.config` — no hardcoded paths.

In [ ]:
import sys

import numpy as np

from ml4b.data.features import extract_features
from ml4b.data.loader import EXERCISE_MAPPING, load_recofit_raw
from ml4b.data.splitting import subject_based_split
from ml4b.data.windowing import apply_sliding_window
from ml4b.utils.config import (
    DATA_PROCESSED,
    DATA_RAW,
    PROJECT_ROOT,
    find_project_root,
)

# Verify the project root resolved by config.py matches the marker-based
# lookup — guards against PYTHONPATH oddities in editable installs.
assert PROJECT_ROOT == find_project_root(), (
    f"PROJECT_ROOT mismatch: config={PROJECT_ROOT}, marker={find_project_root()}"
)

print(f"Python:        {sys.version.split()[0]}")
print(f"Project root:  {PROJECT_ROOT}")
print(f"DATA_RAW:      {DATA_RAW}")
print(f"DATA_PROCESSED: {DATA_PROCESSED}")

## 2. Load Raw Data

`load_recofit_raw()` parses the 2.5 GB `.mat` file (1–2 minutes) and returns one row per time sample, already filtered to our 6 target classes via `EXERCISE_MAPPING` (see ADR-005). We print the resulting shape and the per-class sample count to confirm coverage.

In [ ]:
mat_file = DATA_RAW / "recofit" / "exercise_data.50.0000_singleonly.mat"
print(f"Loading {mat_file.name} (this takes ~1-2 minutes)...")
raw_df = load_recofit_raw(mat_file)

print(f"\nRaw DataFrame shape: {raw_df.shape}")
print(f"Columns:             {list(raw_df.columns)}")
print(f"Subjects:            {raw_df['subject_id'].nunique()}")
print(
    f"Target classes:      {raw_df['exercise_name'].nunique()} "
    f"({sorted(raw_df['exercise_name'].unique())})"
)

print("\nSamples per target class:")
print(raw_df["exercise_name"].value_counts().to_string())

print(
    f"\nMapped RecoFit labels: {len(EXERCISE_MAPPING)} "
    f"→ {len(set(EXERCISE_MAPPING.values()))} target classes"
)

## 3. Apply Sliding Windows

Each continuous recording is segmented into 2-second windows (100 samples at 50 Hz) with 50% overlap. Windows never cross subject, exercise, or recording boundaries — see ADR-006 for the parameter rationale.

In [ ]:
windows_df = apply_sliding_window(
    raw_df, window_size=100, overlap=0.5, sampling_rate=50
)

print(f"Total windows: {len(windows_df):,}")
print(f"Columns:       {list(windows_df.columns)}")
print("\nWindows per class:")
print(windows_df["exercise_name"].value_counts().to_string())
print("\nWindows per subject (head):")
print(
    windows_df.groupby("subject_id")
    .size()
    .sort_values(ascending=False)
    .head()
    .to_string()
)

## 4. Extract Features

Per window: 7 statistical features × 6 axes (42), plus 3 magnitude features, plus 2 FFT-based features = **47 numeric features**. Each row of the resulting DataFrame is a single training sample for Phase 4.

In [ ]:
features_df = extract_features(windows_df)

# Separate identifier columns from numeric feature columns for clarity.
id_cols = ["subject_id", "exercise_name", "window_id"]
feature_cols = [c for c in features_df.columns if c not in id_cols]

print(f"Feature matrix shape: {features_df.shape}")
print(f"Number of features:   {len(feature_cols)}")
print(f"\nFeature names ({len(feature_cols)}):")
for i, name in enumerate(feature_cols):
    print(f"  {i:2d}: {name}")

print("\nFirst 3 rows (selected columns):")
preview_cols = id_cols + feature_cols[:5] + ["dominant_frequency", "spectral_energy"]
print(features_df[preview_cols].head(3).to_string(index=False))

## 5. Subject-Based Train / Validation / Test Split

Subjects — not individual rows — are partitioned 70 / 10 / 20. Every window from a given subject is assigned entirely to one split. This eliminates leakage from a person-specific motion signature and mirrors the real deployment scenario (Apple Watch on a brand-new user). See ADR-007.

In [ ]:
train_df, val_df, test_df = subject_based_split(
    features_df, test_size=0.2, val_size=0.1, random_state=42
)

total = len(features_df)
print("Split sizes (windows):")
print(f"  train: {len(train_df):>6,}  ({len(train_df) / total:.1%})")
print(f"  val:   {len(val_df):>6,}  ({len(val_df) / total:.1%})")
print(f"  test:  {len(test_df):>6,}  ({len(test_df) / total:.1%})")

print("\nSplit sizes (subjects):")
print(f"  train: {train_df['subject_id'].nunique()} subjects")
print(f"  val:   {val_df['subject_id'].nunique()} subjects")
print(f"  test:  {test_df['subject_id'].nunique()} subjects")

## Why Only Undersample the Training Set?

After windowing, the `rest` class accounts for **~88.8% of windows** — a model that always
predicts `rest` would achieve ~89% accuracy, making accuracy a misleading metric.

The fix: cap `rest` in the **training set only** at `2×` the largest exercise class.

Val and test sets keep the original distribution intentionally:
- The model trains on balanced classes → learns all 6 exercises equally.
- Evaluation metrics (val, test) reflect **real-world distribution** → honest generalization estimate.
- Macro-averaged F1 is used as the primary metric — not affected by class imbalance in test.

See `docs/decisions/ADR-008-undersampling-strategy.md` for full rationale.

In [ ]:
from ml4b.data.splitting import undersample_majority_class

# Apply undersampling to training set ONLY.
# IMPORTANT: val and test must keep the original class distribution
# so evaluation metrics reflect real-world conditions.
train_df_balanced = undersample_majority_class(
    train_df,
    majority_class="rest",
    multiplier=2.0,
    random_state=42,
)

print(f"\nTotal train windows BEFORE undersampling: {len(train_df):,}")
print(f"Total train windows AFTER  undersampling: {len(train_df_balanced):,}")
print(
    f"Reduction: {(1 - len(train_df_balanced) / len(train_df)):.1%} fewer training windows"
)

## 6. Save Processed Data

Writes four artefacts under `data/processed/`:

* `train_features.csv`
* `val_features.csv`
* `test_features.csv`
* `feature_names.txt` — newline-separated list of numeric feature columns (single source of truth for Phase 4).

`data/processed/` is in `.gitignore`, so these files stay local — they are reproducible from this notebook in ~3 minutes.

In [ ]:
# Save balanced train + unmodified val and test.
# Val and test keep the real-world class distribution for honest evaluation.
# Ensure the output directory exists; .gitkeep is committed but the directory
# itself might be missing on a fresh clone after a `git clean`.
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

train_path = DATA_PROCESSED / "train_features.csv"
val_path = DATA_PROCESSED / "val_features.csv"
test_path = DATA_PROCESSED / "test_features.csv"
feature_names_path = DATA_PROCESSED / "feature_names.txt"

train_df_balanced.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)
feature_names_path.write_text("\n".join(feature_cols) + "\n")

for p in [train_path, val_path, test_path, feature_names_path]:
    print(f"  wrote {p.relative_to(PROJECT_ROOT)}  ({p.stat().st_size / 1024:.1f} KB)")

## 7. Sanity Checks

Three invariants must hold before we declare Phase 3 done:

1. **No subject overlap** between any pair of splits.  
2. **All 6 classes** present in every split.  
3. **No NaN / inf** in numeric feature columns.

In [ ]:
train_subj = set(train_df_balanced["subject_id"])
val_subj = set(val_df["subject_id"])
test_subj = set(test_df["subject_id"])

# 1. Disjoint-subject check — the entire point of subject-based splitting.
assert train_subj.isdisjoint(val_subj), "Subject leak between train and val!"
assert train_subj.isdisjoint(test_subj), "Subject leak between train and test!"
assert val_subj.isdisjoint(test_subj), "Subject leak between val and test!"
print("No subject overlap between any pair of splits.")

# 2. Class coverage per split — fewer than 6 classes in any split would make
# per-class metrics undefined and likely indicate an unlucky shuffle.
print("\nClass distribution per split:")
for name, split in [("train", train_df_balanced), ("val", val_df), ("test", test_df)]:
    counts = split["exercise_name"].value_counts().sort_index()
    print(f"  {name}: {len(counts)} classes — {counts.to_dict()}")

# 3. NaN / inf check — extract_features should produce finite floats, but a
# constant-signal window or empty FFT could in principle yield 0/0.
for name, split in [("train", train_df_balanced), ("val", val_df), ("test", test_df)]:
    arr = split[feature_cols].to_numpy()
    n_nan = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    print(f"  {name}: NaN={n_nan}, inf={n_inf}")
    assert n_nan == 0 and n_inf == 0, f"Found NaN/inf in {name} features!"

print("\nAll sanity checks passed — Phase 3 complete.")

## 8. Summary of Phase 3 Findings

_Fill in the values below after running all cells above._

| Metric | Value |
|--------|-------|
| Raw samples loaded | _e.g. 8_500_000_ |
| Target classes | 6 (bicep_curl, shoulder_press, squat, tricep_extension, lateral_raise, rest) |
| Total windows | _e.g. 85_000_ |
| Features per window | 47 |
| Subjects (train / val / test) | _e.g. 65 / 9 / 19_ |
| Windows (train / val / test) | _e.g. 59500 / 8500 / 17000_ |
| Subject overlap between splits | 0 ✅ |
| NaN / inf in features | 0 ✅ |

### Decisions made
* **Window size = 2 s, overlap = 50%** — see `docs/decisions/ADR-006-sliding-window-parameters.md`.
* **Subject-based 70/10/20 split** — see `docs/decisions/ADR-007-subject-based-train-test-split.md`.

### Hand-off to Phase 4
Phase 4 (Modeling) reads exactly:
* `data/processed/train_features.csv`
* `data/processed/val_features.csv`
* `data/processed/test_features.csv`
* `data/processed/feature_names.txt`

No re-loading of the `.mat` file is needed.